# Version Specification Analysis (Refined RQ3)

**Research Question**: When AI agents introduce new libraries, do they specify versions?

This notebook provides a more accurate analysis by distinguishing:
1. **Modified files**: Adding libraries to existing dependency files (agents choosing to add specific libraries)
2. **Added files**: Creating new dependency files (project initialization)

## Context from Prior Work

**Raj & Costa (MSR 2024)**: "The role of library versions in Developer-ChatGPT conversations"
- Found only **9.67%** of ChatGPT conversations mention library versions
- Dataset: 486 library-related conversations from DevGPT
- Context: Casual developer-AI interactions

## Our Question

Do AI agents behave differently in **production code** (merged PRs) vs conversations?

In [ ]:
import sys

sys.path.append("..")

import json
import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

data_dir = Path("../data")
output_dir = Path("../output")

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## Load Analysis Results

In [ ]:
# Load results for each language
results_by_lang = {}

for lang in ["go", "python", "typescript"]:
    with open(output_dir / f"{lang}_library_usage.json", "r") as f:
        results_by_lang[lang.title()] = json.load(f)

print("Loaded results:")
for lang, results in results_by_lang.items():
    print(f"  {lang}: {len(results):,} PRs")

## Critical Distinction: Modified vs Added Files

Our PR analyzer tracks two flags:
- `dep_file_modified`: Agent modified an existing dependency file (adding to established project)
- `dep_file_added`: Agent created a new dependency file (project initialization)

For comparison to prior work, we should focus on **modified files** where agents are actively choosing to add specific libraries.

In [ ]:
# Analyze modified vs added breakdown
print("=" * 70)
print("BREAKDOWN: Modified vs Added Dependency Files")
print("=" * 70)

breakdown_data = []

for lang, results in results_by_lang.items():
    # Separate by file operation type
    prs_modifying = [r for r in results if r["dep_file_modified"]]
    prs_adding = [r for r in results if r["dep_file_added"]]

    print(f"\n{lang}:")
    print(f"  PRs modifying existing dep files: {len(prs_modifying):,}")
    print(f"  PRs adding new dep files: {len(prs_adding):,}")
    print(
        f"  Ratio: {len(prs_modifying) / (len(prs_adding) or 1):.1f}x more modified than added"
    )

    breakdown_data.append(
        {"Language": lang, "Modified": len(prs_modifying), "Added": len(prs_adding)}
    )

df_breakdown = pd.DataFrame(breakdown_data)

In [ ]:
# Visualize breakdown
fig, ax = plt.subplots(figsize=(10, 6))
df_breakdown.set_index("Language")[["Modified", "Added"]].plot(
    kind="bar", ax=ax, width=0.7
)
plt.ylabel("Number of PRs")
plt.xlabel("Language")
plt.title("Dependency File Operations: Modified vs Added")
plt.legend(title="Operation", labels=["Modified Existing", "Added New"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(output_dir / "dep_file_operations.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"\nSaved: {output_dir / 'dep_file_operations.png'}")

## RQ3a: Version Specs When Modifying Existing Dependencies

In [ ]:
# Calculate version specification rates for MODIFIED files only
print("=" * 70)
print("RQ3a: VERSION SPECS IN MODIFIED DEPENDENCY FILES")
print("=" * 70)
print("\n(Most relevant for comparison to prior work)\n")

modified_stats = []

for lang, results in results_by_lang.items():
    # Focus ONLY on PRs that modified existing dep files
    prs_modifying = [r for r in results if r["dep_file_modified"]]

    if not prs_modifying:
        continue

    # Sum up version statistics
    total_with_ver = sum(r["libs_with_version"] for r in prs_modifying)
    total_without_ver = sum(r["libs_without_version"] for r in prs_modifying)
    total = total_with_ver + total_without_ver

    pct_with_ver = 100 * total_with_ver / total if total > 0 else 0

    print(f"{lang}:")
    print(f"  PRs modifying deps: {len(prs_modifying):,}")
    print(f"  Libs with version: {total_with_ver:,}")
    print(f"  Libs without version: {total_without_ver:,}")
    print(f"  % with version: {pct_with_ver:.1f}%")
    print()

    modified_stats.append(
        {
            "Language": lang,
            "PRs": len(prs_modifying),
            "With Version": total_with_ver,
            "Without Version": total_without_ver,
            "% With Version": pct_with_ver,
        }
    )

df_modified_stats = pd.DataFrame(modified_stats)
print("\nSummary Table:")
print(df_modified_stats.to_string(index=False))

## RQ3b: Version Specs When Initializing Projects

In [ ]:
# Calculate version specification rates for ADDED files
print("=" * 70)
print("RQ3b: VERSION SPECS IN NEWLY CREATED DEPENDENCY FILES")
print("=" * 70)
print("\n(Project initialization)\n")

added_stats = []

for lang, results in results_by_lang.items():
    # Focus on PRs that added new dep files
    prs_adding = [r for r in results if r["dep_file_added"]]

    if not prs_adding:
        print(f"{lang}: No PRs added new dependency files")
        continue

    # Sum up version statistics
    total_with_ver = sum(r["libs_with_version"] for r in prs_adding)
    total_without_ver = sum(r["libs_without_version"] for r in prs_adding)
    total = total_with_ver + total_without_ver

    pct_with_ver = 100 * total_with_ver / total if total > 0 else 0

    print(f"{lang}:")
    print(f"  PRs adding new deps: {len(prs_adding):,}")
    print(f"  Libs with version: {total_with_ver:,}")
    print(f"  Libs without version: {total_without_ver:,}")
    print(f"  % with version: {pct_with_ver:.1f}%")
    print()

    added_stats.append(
        {
            "Language": lang,
            "PRs": len(prs_adding),
            "With Version": total_with_ver,
            "Without Version": total_without_ver,
            "% With Version": pct_with_ver,
        }
    )

if added_stats:
    df_added_stats = pd.DataFrame(added_stats)
    print("\nSummary Table:")
    print(df_added_stats.to_string(index=False))

## Comparison: Modified vs Added Files

In [ ]:
# Create comparison visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Modified files (left)
languages = df_modified_stats["Language"]
modified_pcts = df_modified_stats["% With Version"]

ax1.bar(languages, modified_pcts, color="steelblue", alpha=0.8)
ax1.axhline(
    y=9.67,
    color="red",
    linestyle="--",
    linewidth=2,
    label="ChatGPT Conversations (9.67%)",
)
ax1.set_ylabel("% with Version Specification")
ax1.set_xlabel("Language")
ax1.set_title("Modified Dependency Files\n(Adding to Existing Projects)")
ax1.set_ylim(0, 105)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Added files (right) - if data exists
if added_stats:
    added_langs = df_added_stats["Language"]
    added_pcts = df_added_stats["% With Version"]

    ax2.bar(added_langs, added_pcts, color="coral", alpha=0.8)
    ax2.axhline(
        y=9.67,
        color="red",
        linestyle="--",
        linewidth=2,
        label="ChatGPT Conversations (9.67%)",
    )
    ax2.set_ylabel("% with Version Specification")
    ax2.set_xlabel("Language")
    ax2.set_title("Added Dependency Files\n(Project Initialization)")
    ax2.set_ylim(0, 105)
    ax2.legend()
    ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / "version_specs_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"\nSaved: {output_dir / 'version_specs_comparison.png'}")

## Version Operator Analysis

In [ ]:
# Analyze which version operators are used by language
print("=" * 70)
print("VERSION OPERATOR USAGE BY LANGUAGE")
print("=" * 70)

operator_data = []

for lang, results in results_by_lang.items():
    # Focus on modified files for consistency
    prs_modifying = [r for r in results if r["dep_file_modified"]]

    # Aggregate version operators
    all_operators = Counter()
    for pr in prs_modifying:
        ops = pr.get("version_operators", {})
        all_operators.update(ops)

    print(f"\n{lang}:")
    if all_operators:
        for op, count in all_operators.most_common():
            pct = 100 * count / sum(all_operators.values())
            print(f"  {op:5s} {count:4d} ({pct:5.1f}%)")
            operator_data.append(
                {"Language": lang, "Operator": op, "Count": count, "Percentage": pct}
            )
    else:
        print("  No version operators found")

In [ ]:
# Visualize operator distribution
if operator_data:
    df_operators = pd.DataFrame(operator_data)

    # Pivot for grouped bar chart
    df_pivot = df_operators.pivot_table(
        index="Language", columns="Operator", values="Percentage", fill_value=0
    )

    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    df_pivot.plot(kind="bar", ax=ax, width=0.8)
    plt.ylabel("Percentage of Version Specifications")
    plt.xlabel("Language")
    plt.title("Version Operator Distribution by Language")
    plt.legend(title="Operator", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(output_dir / "version_operators.png", dpi=300, bbox_inches="tight")
    plt.show()

    print(f"\nSaved: {output_dir / 'version_operators.png'}")

## Key Findings Summary

In [ ]:
print("=" * 70)
print("KEY FINDINGS FOR PAPER")
print("=" * 70)

print("\n1. STARK CONTRAST WITH PRIOR WORK")
print("-" * 70)
print("Prior work (Raj & Costa, MSR 2024):")
print("  - ChatGPT conversations: 9.67% mention versions")
print("  - Context: Casual developer-AI interactions")
print("\nOur findings (agent PRs, modified files):")
for _, row in df_modified_stats.iterrows():
    print(f"  - {row['Language']:12s}: {row['% With Version']:5.1f}% specify versions")
print("\nGap: 8-10x higher in production code!")

print("\n2. CONTEXT MATTERS")
print("-" * 70)
print("Modifying existing projects (agents actively choosing libraries):")
for _, row in df_modified_stats.iterrows():
    print(f"  - {row['Language']:12s}: {row['% With Version']:5.1f}%")

if added_stats:
    print("\nInitializing new projects (project setup):")
    for _, row in df_added_stats.iterrows():
        print(f"  - {row['Language']:12s}: {row['% With Version']:5.1f}%")
    print("\nImplication: Agents adapt behavior based on context")

print("\n3. LANGUAGE ECOSYSTEM EFFECTS")
print("-" * 70)
print("TypeScript/Go: Ecosystem enforces versioning → 100%")
print("Python: More flexible → 84% (modified), 33% (added)")
print("Implication: Tool design influences agent behavior")

print("\n4. VERSION OPERATOR PREFERENCES")
print("-" * 70)
if operator_data:
    python_ops = df_operators[df_operators["Language"] == "Python"].nlargest(3, "Count")
    ts_ops = df_operators[df_operators["Language"] == "TypeScript"].nlargest(3, "Count")

    if not python_ops.empty:
        print("Python prefers:")
        for _, row in python_ops.iterrows():
            print(f"  - {row['Operator']:5s} ({row['Percentage']:.1f}%)")

    if not ts_ops.empty:
        print("TypeScript prefers:")
        for _, row in ts_ops.iterrows():
            print(f"  - {row['Operator']:5s} ({row['Percentage']:.1f}%)")

    print("\nReflects ecosystem conventions: exact (==) vs compatible (^)")

## Export Results for Paper

In [ ]:
# Save comprehensive results
version_analysis_results = {
    "modified_files": df_modified_stats.to_dict("records"),
    "added_files": df_added_stats.to_dict("records") if added_stats else [],
    "version_operators": operator_data,
    "comparison_to_prior_work": {
        "raj_costa_msr2024": {
            "context": "ChatGPT conversations",
            "version_mention_rate": 9.67,
            "unit": "percent",
        },
        "our_findings_modified": {
            "context": "Agent PRs modifying existing dependency files",
            "rates_by_language": {
                row["Language"]: row["% With Version"]
                for _, row in df_modified_stats.iterrows()
            },
        },
        "gap": "8-10x higher in production code",
    },
    "key_insights": [
        "Agents are 8-10x more diligent about versions in PRs vs conversations",
        "Context matters: 84-100% (modified) vs 33-100% (added)",
        "Language ecosystems influence behavior",
        "Python prefers exact versions (==), TypeScript prefers compatible (^)",
    ],
}

with open(output_dir / "version_specification_analysis.json", "w") as f:
    json.dump(version_analysis_results, f, indent=2)

print("\nVersion specification analysis complete!")
print(f"Results saved to: {output_dir / 'version_specification_analysis.json'}")

print("\nGenerated visualizations:")
print(f"  1. {output_dir / 'dep_file_operations.png'}")
print(f"  2. {output_dir / 'version_specs_comparison.png'}")
print(f"  3. {output_dir / 'version_operators.png'}")

## Paper Narrative (Draft)

### RQ3: Do AI agents specify library versions?

**Context**: Previous work by Raj and Costa (MSR 2024) found that ChatGPT mentions library versions in only 9.67% of developer conversations, raising concerns about AI's attention to dependency management.

**Our Investigation**: We examined version specification behavior in agent-authored PRs, distinguishing between:
1. **Modified files**: Agents adding libraries to existing projects (actively choosing specific libraries)
2. **Added files**: Agents initializing new projects (project setup)

**Finding 1 - Production vs Conversations**: When modifying existing dependency files, agents specify versions at dramatically higher rates:
- **Go**: 100% (203/203 libraries)
- **TypeScript**: 100% (2,584/2,584 libraries)
- **Python**: 83.9% (759/905 libraries)

This represents an **8-10x improvement** over casual conversations, suggesting agents adapt their behavior based on context: production code contributions require more rigor than exploratory conversations.

**Finding 2 - Context Dependence**: Behavior varies significantly between modifying existing projects and initializing new ones. Python shows the largest gap (84% → 33%), while TypeScript maintains 100% in both cases. This variation suggests:
- Ecosystem tooling influences behavior (npm enforces versions)
- Agents may be more careful when modifying established codebases
- Language conventions affect version specification practices

**Finding 3 - Operator Preferences**: Agents follow language-specific conventions:
- **Python**: Exact versions (`==`) dominate (88% of specifications)
- **TypeScript**: Compatible ranges (`^`) preferred (73% of specifications)
- **Go**: Module system enforces complete version specifications

**Implication**: These findings challenge concerns about AI carelessness with dependencies. While agents are less precise in casual conversations, they demonstrate high diligence in production code, adapting to ecosystem norms and project context.